# 04 · Hit vs Miss Coding Axis (supervised DV)

Supervised decision-variable pipeline for the change-detection dataset.
Unlike `02` (unsupervised PCA trajectories), this fits a **cross-validated,
regularized hit-vs-miss axis** and projects single trials onto it to get a
decision variable DV(t), then runs the controls that guard against the
"internal state = arousal relabeled" failure mode.

Design constraints baked in:
- Change trials only, aligned to change-flash onset → sensory input held constant.
- Fit window ends *before* most licks, so the axis isn't reading lick execution
  (misses have no lick by construction).
- Everything is **per `ophys_experiment_id`** (a population = one experiment);
  pick one from the trade-off table in section 2 (no pilot session is ideal).
- Nothing is fit or z-scored on the trials it is evaluated on (no leakage).


## 1 · Environment + data (same Colab loader as `02`)

In [ ]:
import importlib.util
import subprocess
import sys

def ensure_package(import_name, pip_name=None):
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or import_name])

ensure_package("pyarrow")
ensure_package("gdown")
ensure_package("sklearn", "scikit-learn")
ensure_package("scipy")

from pathlib import Path
import gdown
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
GDRIVE_URL = "https://drive.google.com/file/d/1NGZE-uG7GTJnLeR2P6ip98249lJeYC8d/view?usp=share_link"
PARQUET_PATH = Path("data/allen_visp_sst_vip_slc17a7_pilot.parquet")

PARQUET_PATH.parent.mkdir(parents=True, exist_ok=True)
if not PARQUET_PATH.exists():
    downloaded_path = gdown.download(url=GDRIVE_URL, output=str(PARQUET_PATH), quiet=False, fuzzy=True)
    if downloaded_path is None:
        raise RuntimeError("Download failed. Check that the Drive file is shared with anyone who has the link.")

print(f"Using parquet: {PARQUET_PATH.resolve()}")

## 2 · Select experiment and build `all_activity`

One population = one `ophys_experiment_id`. **No session in this pilot is ideal**:
the excitatory line (Slc17a7) has the neurons but too few miss trials to
cross-validate, while the interneuron lines (Sst/Vip) have trials but tiny
populations. The table below prints both axes of that trade-off — `neurons`
(population size) and `min_class` (the CV-limiting count, = min(hit, miss)).

Default selection maximizes `min_class` so cross-validation is stable. Set
`EXPERIMENT_ID` by hand to force a specific session/cre line. For a real
excitatory analysis you need more sessions — pull a cohort with notebook `03`.


In [ ]:
data = pd.read_parquet(PARQUET_PATH)
required = {"stimulus_presentations_id", "cell_specimen_id", "trace", "trace_timestamps",
            "is_change", "behavior_outcome", "ophys_experiment_id",
            "mean_pupil_area", "mean_running_speed", "response_latency"}
missing = sorted(required - set(data.columns))
if missing:
    raise ValueError(f"Parquet is missing Allen analysis column(s): {missing}")

change_data = data[data["is_change"].fillna(False).astype(bool)].copy()
change_data = change_data[change_data["behavior_outcome"].isin(["hit", "miss"])]

trial_counts = (change_data.drop_duplicates(["ophys_experiment_id", "stimulus_presentations_id"])
                .groupby(["ophys_experiment_id", "behavior_outcome"]).size().unstack(fill_value=0)
                .reindex(columns=["hit", "miss"], fill_value=0))
eligible = trial_counts[(trial_counts["hit"] > 0) & (trial_counts["miss"] > 0)]
if eligible.empty:
    raise ValueError("No experiment in this parquet contains both hit and miss change trials.")

def _n_common_cells(eid):
    ed = change_data[change_data.ophys_experiment_id == eid]
    sets = [set(g["cell_specimen_id"].dropna()) for _, g in ed.groupby("stimulus_presentations_id")]
    return len(set.intersection(*sets)) if sets else 0

summary = eligible.copy()
summary["cre_line"]  = [change_data.loc[change_data.ophys_experiment_id == e, "cre_line"].iloc[0]
                        for e in summary.index]
summary["neurons"]   = [_n_common_cells(e) for e in summary.index]
summary["min_class"] = summary[["hit", "miss"]].min(axis=1)   # cross-validation is limited by this
display(summary)

# Set EXPERIMENT_ID to force a session/cre line; None => max min_class (most CV-stable).
EXPERIMENT_ID = None
experiment_id = EXPERIMENT_ID if EXPERIMENT_ID is not None else summary["min_class"].idxmax()
experiment_data = change_data[change_data["ophys_experiment_id"] == experiment_id].copy()
row = summary.loc[experiment_id]
print(f"Selected {experiment_id}: cre={row.cre_line}, neurons={row.neurons}, "
      f"hits={row.hit}, miss={row.miss}, minority class={'hit' if row.hit < row.miss else 'miss'}")
if row.neurons < 30:
    print(f"  WARNING: only {row.neurons} neurons — the coding axis is low-dimensional and two "
          f"random axes align easily; lean on the shuffle null in section 8.")

In [ ]:
# trial metadata in stable stimulus order (same as 02 cell 4)
trial_meta = (experiment_data[["stimulus_presentations_id", "behavior_outcome"]]
              .drop_duplicates("stimulus_presentations_id").sort_values("stimulus_presentations_id"))
trial_ids = trial_meta["stimulus_presentations_id"].tolist()

cell_sets = [set(g["cell_specimen_id"].dropna()) for _, g in experiment_data.groupby("stimulus_presentations_id")]
common_cells = sorted(set.intersection(*cell_sets))
if not common_cells:
    raise ValueError("The selected trials have no common cells.")

activity = []
for trial_id in trial_ids:
    rows = (experiment_data[experiment_data["stimulus_presentations_id"] == trial_id]
            .set_index("cell_specimen_id").loc[common_cells])
    activity.append(np.stack([np.asarray(t, dtype=float) for t in rows["trace"]]))

all_activity = np.stack(activity)                       # trials x neurons x time
rel_time = np.asarray(experiment_data.iloc[0]["trace_timestamps"], dtype=float)
if all_activity.shape[2] != len(rel_time):
    raise ValueError("Trace and timestamp lengths do not match.")

outcomes = trial_meta["behavior_outcome"].to_numpy()
n_trials, n_neurons, n_time = all_activity.shape
print("all_activity (trials, neurons, time):", all_activity.shape)
print("hits:", int((outcomes == "hit").sum()), " miss:", int((outcomes == "miss").sum()))

## 3 · Per-trial covariates (for the controls)

One scalar per trial, in the same order as `all_activity`. `response_latency` is
NaN on miss trials by construction — it only exists where there was a lick.


In [ ]:
trial_cov = (experiment_data.drop_duplicates("stimulus_presentations_id")
             .set_index("stimulus_presentations_id")
             .loc[trial_ids, ["mean_pupil_area", "mean_running_speed", "response_latency"]])
pupil       = trial_cov["mean_pupil_area"].to_numpy(float)
running     = trial_cov["mean_running_speed"].to_numpy(float)
rt          = trial_cov["response_latency"].to_numpy(float)
trial_index = np.argsort(np.argsort(trial_ids)).astype(float)   # session-order proxy for slow drift
y = (outcomes == "hit").astype(int)                             # 1 = hit, 0 = miss

print("median hit RT (s):", np.nanmedian(rt[y == 1]))

## 4 · Baseline-correct + NaN handling

Subtract each trial's own pre-change baseline so the axis reads the *change response*,
not a tonic offset that would correlate with arousal. Frames dropped by the ophys
pipeline become 0 after baseline (they are already trial-local).


In [ ]:
base = rel_time < 0
A = all_activity - np.nanmean(all_activity[:, :, base], axis=2, keepdims=True)
A = np.nan_to_num(A, nan=0.0)                                   # trials x neurons x time

### 4b · Optional: regress arousal out of the population *before* fitting

The section-8 cosine only *diagnoses* arousal leakage after the fact. This step
*removes* it up front: from every (neuron, timepoint) it subtracts the linear
component predictable from that trial's pupil + running, so the axis is fit on
arousal-residual activity and the DV is arousal-free by construction.

Leakage note: the residualization uses only the **covariates**, never the outcome
label, so fitting it on all trials does not leak hit/miss into held-out decoding.
(A fully strict version refits the covariate betas inside each CV fold; the global
fit here is standard and adequate because `y` never enters it.) Set the flag False
to compare against raw activity — the section-8 cosine should collapse toward 0
when this is on, which is the check that it worked.


In [ ]:
RESIDUALIZE_AROUSAL = True

A_raw = A.copy()
if RESIDUALIZE_AROUSAL:
    cov = np.column_stack([pupil, running]).astype(float)
    col_mean = np.nanmean(cov, axis=0)
    cov = np.where(np.isfinite(cov), cov, col_mean)            # mean-impute blinks/missing
    cov = (cov - cov.mean(0)) / cov.std(0)
    Z = np.column_stack([np.ones(n_trials), cov])             # intercept + pupil + running
    A2d = A.reshape(n_trials, -1)
    beta, *_ = np.linalg.lstsq(Z, A2d, rcond=None)            # per (neuron,time) regression
    A = (A2d - Z @ beta).reshape(n_trials, n_neurons, n_time)
    print(f"Arousal residualized out. Population variance removed: "
          f"{1 - A.var() / A_raw.var():.3f}")
else:
    print("RESIDUALIZE_AROUSAL=False - using raw baseline-corrected activity.")

### 4c · Optional: regress slow session drift out of the population

`trial_index` was the single largest predictor of outcome — hit rate decays across
the session (engagement drift, satiety, bleaching), and that decay is confounded
with the internal state we want to measure. This removes a **smooth** function of
session position (a low-order polynomial basis) from every (neuron, timepoint), the
exact analog of the arousal step.

The decisive question it answers: does the DV effect **survive** drift removal?
- DV still separates hit/miss afterward → the axis tracks an *instantaneous* state,
  not just slow session decay (this is the result you want).
- DV collapses → what looked like an engagement gate was largely session drift.

Keep the basis low-order so it removes smooth drift, not trial-to-trial signal. It
uses only trial position (never the outcome), so there is no label leakage.


In [ ]:
# diagnostic: rolling hit rate across the session makes the drift visible
order = np.argsort(trial_index)
w = min(20, n_trials // 3)
roll = np.convolve(y[order].astype(float), np.ones(w) / w, mode="valid")
fig, ax = plt.subplots(figsize=(7, 2.6))
ax.plot(np.arange(len(roll)), roll)
ax.set(xlabel="trial (session order)", ylabel=f"hit rate ({w}-trial roll)",
       title="Engagement drift across the session")
plt.show()

In [ ]:
REMOVE_DRIFT = True
DRIFT_DF = 3                                                   # polynomial order of the drift basis (keep small)

if REMOVE_DRIFT:
    ti = (trial_index - trial_index.mean()) / trial_index.std()
    B = np.vander(ti, DRIFT_DF + 1, increasing=True)          # [1, ti, ti^2, ti^3]
    A2d = A.reshape(n_trials, -1)
    beta, *_ = np.linalg.lstsq(B, A2d, rcond=None)
    var_before = A.var()
    A = (A2d - B @ beta).reshape(n_trials, n_neurons, n_time)

    from scipy.stats import pointbiserialr
    r_drift = pointbiserialr(y, trial_index).correlation
    print(f"Removed degree-{DRIFT_DF} session drift. Population variance removed: "
          f"{1 - A.var() / var_before:.3f}")
    print(f"Behavioral outcome-vs-trial_index correlation: r={r_drift:+.3f} "
          f"(the confound the neural residualization mirrors).")
    print("Now re-run sections 5-8: if the DV separation and the section-8 DV "
          "coefficient hold up, the axis tracks instantaneous state, not drift.")
else:
    print("REMOVE_DRIFT=False - session drift left in the population.")

## 5 · Cross-validated shrinkage axis → out-of-fold DV(t)

Two non-negotiables:
- **Regularize** — few trials in the minority class (in this pilot session that is
  *hits*, not misses); unregularized LDA/logistic decodes perfectly in-sample and
  nothing out-of-sample.
- **No leakage** — the `StandardScaler` and the classifier are fit inside each
  fold and only ever *evaluated* on held-out trials.

`FIT_WIN` ends at 0.3 s on purpose: fitting later folds lick execution into the axis,
and since misses never lick, that would manufacture a hit/miss difference. Logistic
regression is the default (drops LDA's equal-covariance assumption, which is exactly
what an "engaged vs disengaged" noise difference would violate); a shrinkage-LDA
swap is provided in the next cell.


In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

FIT_WIN = (0.0, 0.30)
fit_mask = (rel_time >= FIT_WIN[0]) & (rel_time <= FIT_WIN[1])

def make_axis():
    return make_pipeline(
        StandardScaler(),
        LogisticRegression(penalty="l2", C=0.1, class_weight="balanced", max_iter=2000),
    )

N_SPLITS = min(5, int(y.sum()), int((1 - y).sum()))            # never more folds than the minority class
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=0)

DV = np.full((n_trials, n_time), np.nan)                       # out-of-fold decision function
axes_w = []                                                    # per-fold weight vectors (fit window)
for tr, te in skf.split(np.zeros(n_trials), y):
    clf = make_axis().fit(A[tr][:, :, fit_mask].mean(axis=2), y[tr])
    axes_w.append(clf[-1].coef_.ravel())
    for t in range(n_time):
        DV[te, t] = clf.decision_function(A[te][:, :, t])

print(f"{N_SPLITS}-fold out-of-fold DV computed. axis dim = {axes_w[0].shape[0]} neurons")

In [ ]:
# OPTIONAL robustness swap: shrinkage LDA instead of logistic.
# from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
# def make_axis():
#     return make_pipeline(StandardScaler(),
#                          LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto"))

## 6 · DV(t) by outcome + the initial-conditions test

Reading against the mechanistic taxonomy:
- miss ≈ flat at the hit baseline → **engagement gate upstream of accumulation**
- miss ramps but shallower → **reduced drift**
- miss ramps normally from a lower start → **state-dependent shift in initial conditions**

The baseline (pre-change) projection is the direct test of that last case.


In [ ]:
from scipy.stats import mannwhitneyu

dv_hit  = np.nanmean(DV[y == 1], axis=0)
dv_miss = np.nanmean(DV[y == 0], axis=0)
se_hit  = np.nanstd(DV[y == 1], axis=0) / np.sqrt((y == 1).sum())
se_miss = np.nanstd(DV[y == 0], axis=0) / np.sqrt((y == 0).sum())

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(rel_time, dv_hit, label="hit"); ax.fill_between(rel_time, dv_hit-se_hit, dv_hit+se_hit, alpha=.2)
ax.plot(rel_time, dv_miss, label="miss"); ax.fill_between(rel_time, dv_miss-se_miss, dv_miss+se_miss, alpha=.2)
ax.axvline(0, color="k", ls="--", lw=1); ax.axvspan(*FIT_WIN, color="grey", alpha=.12, label="fit window")
ax.set(xlabel="time from change onset (s)", ylabel="DV (held-out decision fn)",
       title="Decision variable: hit vs miss")
ax.legend(); plt.show()

baseline_dv = np.nanmean(DV[:, base], axis=1)
p_base = mannwhitneyu(baseline_dv[y == 1], baseline_dv[y == 0]).pvalue
print(f"pre-change baseline DV — hit vs miss Mann-Whitney p = {p_base:.3g}")
print(f"  mean baseline DV  hit={baseline_dv[y==1].mean():+.3f}  miss={baseline_dv[y==0].mean():+.3f}")

## 7 · Cross-temporal decoding (train t, test t′)

Separates a **static** code (stable off-diagonal band → consistent with an
initial-condition / gate story) from a **dynamic evolving** code (accuracy hugs
the diagonal → consistent with accumulation/drift). Cross-validated so the
diagonal isn't trivially 1.


In [ ]:
CT = np.zeros((n_time, n_time)); CT_n = np.zeros((n_time, n_time))
for tr, te in skf.split(np.zeros(n_trials), y):
    for i in range(n_time):
        clf = make_axis().fit(A[tr][:, :, i], y[tr])
        pred = clf.predict(A[te][:, :, :].transpose(0, 2, 1).reshape(-1, n_neurons))
        pred = pred.reshape(len(te), n_time)
        CT[i] += (pred == y[te][:, None]).sum(axis=0); CT_n[i] += len(te)
CT = CT / CT_n

fig, ax = plt.subplots(figsize=(5.5, 4.5))
im = ax.imshow(CT, origin="lower", extent=[rel_time[0], rel_time[-1], rel_time[0], rel_time[-1]],
               vmin=0.5, vmax=1.0, cmap="magma", aspect="auto")
ax.set(xlabel="test time (s)", ylabel="train time (s)", title="Cross-temporal decoding accuracy")
ax.axhline(0, color="w", ls=":", lw=.8); ax.axvline(0, color="w", ls=":", lw=.8)
fig.colorbar(im, label="accuracy"); plt.show()

## 8 · Arousal / drift geometry control — "is it just arousal relabeled?"

Two checks:
1. **Axis alignment.** Fit a *pupil* axis (high vs low pupil) from the same
   population and measure its cosine with the hit/miss axis. Near ±1 means the
   decision axis and the arousal axis are the same direction. With
   `RESIDUALIZE_AROUSAL=True` (section 4b) this cosine should collapse toward 0 —
   that is the confirmation the residualization actually removed the shared
   direction, not just the after-the-fact diagnostic it is when residualization is off.
2. **Partial test.** Does the DV still predict outcome after pupil, running, and
   trial-index (slow drift) are in the model? If the DV coefficient survives, the
   signal is more than arousal + drift.


In [ ]:
# 1) axis geometry
w_hitmiss = np.mean(axes_w, axis=0)
w_hitmiss /= np.linalg.norm(w_hitmiss)

def decoding_axis(binary_target):
    m = np.isfinite(binary_target)
    clf = make_axis().fit(A[m][:, :, fit_mask].mean(axis=2), binary_target[m].astype(int))
    w = clf[-1].coef_.ravel(); return w / np.linalg.norm(w)

hi_pupil   = (pupil   > np.nanmedian(pupil)).astype(float);   hi_pupil[~np.isfinite(pupil)]     = np.nan
hi_running = (running > np.nanmedian(running)).astype(float); hi_running[~np.isfinite(running)] = np.nan
w_pupil   = decoding_axis(hi_pupil)
w_running = decoding_axis(hi_running)
print(f"cos(hit-miss axis, pupil axis)   = {w_hitmiss @ w_pupil:+.3f}")
print(f"cos(hit-miss axis, running axis) = {w_hitmiss @ w_running:+.3f}")
print("  (|cos| near 1 => 'internal state' is that covariate relabeled)")

In [ ]:
# 1b) is that cosine bigger than chance? With few neurons, two axes align easily.
# Null: refit the 'arousal' axis on SHUFFLED pupil labels (breaks the pupil<->neural
# link, keeps the marginal) and recompute |cos| against the fixed hit/miss axis.
rng = np.random.default_rng(0)
obs = abs(w_hitmiss @ w_pupil)
null = np.empty(200)
for k in range(null.size):
    null[k] = abs(w_hitmiss @ decoding_axis(rng.permutation(hi_pupil)))
p_cos = (null >= obs).mean()
q95 = np.quantile(null, .95)
verdict = ("ABOVE the shuffle 95% -> a real shared direction (arousal leakage remains)"
           if obs > q95 else
           "WITHIN the shuffle null -> alignment indistinguishable from chance; no leakage detectable")
print(f"|cos| pupil: observed={obs:.3f}  shuffled mean={null.mean():.3f}  95%={q95:.3f}  p={p_cos:.3f}")
print(f"  -> {verdict}")
print("  Caveat: with few neurons the null is wide, so 'n.s.' can also mean underpowered.")

In [ ]:
# 2) partial test: outcome ~ DV(fit window) + pupil + running + trial_index
import numpy as np
from sklearn.linear_model import LogisticRegression as LR
from sklearn.preprocessing import StandardScaler as SS

dv_fit = np.nanmean(DV[:, fit_mask], axis=1)
design = np.column_stack([dv_fit, pupil, running, trial_index])
names = ["DV", "pupil", "running", "trial_index"]

ok = np.isfinite(design).all(axis=1)
Xd = SS().fit_transform(design[ok])
lr = LR(penalty="l2", C=1.0, class_weight="balanced", max_iter=2000).fit(Xd, y[ok])
coef = pd.Series(lr.coef_.ravel(), index=names).sort_values(key=np.abs, ascending=False)
print(f"n trials with all covariates present: {ok.sum()} / {n_trials}")
print("standardized logistic coefficients (outcome = hit):")
display(coef.to_frame("coef"))
print("If DV stays large and nonzero with pupil/running/trial_index in the model,\n"
      "the coding axis carries outcome information beyond arousal and slow drift.")

## 9 · Caveats to keep in the write-up

- **Lick confound.** `FIT_WIN` avoids it for fitting, but the *late* DV(t) ramp on
  hits still contains lick-related activity absent on misses. For a clean late-window
  claim, truncate each hit trace at its `response_latency` before re-plotting DV(t).
- **Miss count.** With few miss trials, report the cross-validated numbers only and
  bootstrap the axis / baseline test over trials before believing an effect.
- **Modality.** dF/F at 31 Hz is slow — treat ramp *onset/shape* timing cautiously;
  the cross-temporal map is more robust than fine ramp latencies.
- **Per experiment.** Do not pool cre lines into one axis; rerun this notebook per
  `experiment_id`. Slc17a7 (excitatory) is the primary substrate.
